# Allgemeine Analyse: Kongruenz nach Akteur

Erstes Auswertungs-Notebook: **Wie gut stimmen Parolen von Bundesrat, Parlament und Parteien mit dem Volksresultat überein?**

**Was passiert hier?**
- Datensatz mit Kongruenzwerten laden
- Boxplots: Parole (Ja/Nein/…) vs. Kongruenzwert pro Akteur
- Vergleich aller Akteure in einem Plot
- Blog-Grafik `d2_kongruenz_akteur.png` exportieren

**Datengrundlage**
- `data/processed/df_with_positions.csv` (Output von `2_berechnung.ipynb`, Spalten `zustimmung_*`)

**Vorher ausführen**
- `1_data_wrangling.ipynb` → `2_berechnung.ipynb`

**Danach**
- Kein Pflicht-Schritt; parallel: `3b_zeitliche_analyse`, `3c_*`, `3d_thematisch_analyse`
- Blog-Abschnitt „Allgemeine Erkenntnisse“ nutzt den exportierten Plot

## Setup


Projekt-Plots laden; `autoreload` aktualisiert `visualisierungen.py` bei Änderungen.


In [ ]:
%load_ext autoreload
%autoreload 2
from visualisierungen import *


## Daten laden

Kongruenz-Datensatz aus der Pipeline.


Datensatz mit Kongruenzwerten einlesen.


In [ ]:
df = pd.read_csv("../data/processed/df_with_positions.csv")
print(df.head())


Anzahl Abstimmungen mit echtem Volksresultat (Ja-Anteil ≠ 0).


In [ ]:
# Wie viele Abstimmungen haben effektiv stattgefunden?
print(len(df[df["volkja-proz"] != 0]))


## Einzelne Akteure

Stimmt das Volk eher, wenn eine Institution **dafür** oder **dagegen** war? Pro Akteur ein Boxplot.


## Bundesrat

Boxplot Bundesrat: Parole vs. Kongruenz.


In [ ]:
boxplot(df, df["br-pos_label"], df["zustimmung_br-pos"], titel="", xlabel="Bundesratsposition", ylabel="Zustimmung Stimmbevölkerung (%)", farbe=None)


## Bundesversammlung

Boxplot Bundesversammlung.


In [ ]:
boxplot(df, df["bv-pos_label"], df["zustimmung_bv-pos"], titel="", xlabel="Postition Bundesversammlung", ylabel="Zustimmung Stimmbevölkerung (%)", farbe=None)


## SP

Boxplot SP.


In [ ]:
boxplot(df, df["p-sps_label"], df["zustimmung_p-sps"], titel="", xlabel="Position SP", ylabel="Zustimmung Stimmbevölkerung (%)", farbe=None)

### Grüne

Boxplot Grüne.


In [ ]:
boxplot(df, df["p-gps_label"], df["zustimmung_p-gps"], titel="", xlabel="Position Grüne", ylabel="Zustimmung Stimmbevölkerung (%)", farbe=None)

## FDP

Boxplot FDP.


In [ ]:
boxplot(df, df["p-fdp_label"], df["zustimmung_p-fdp"], titel="", xlabel="Position FDP", ylabel="Zustimmung Stimmbevölkerung (%)", farbe=None)

## SVP

Boxplot SVP.


In [ ]:
boxplot(df, df["p-svp_label"], df["zustimmung_p-svp"], titel="", xlabel="Position SVP", ylabel="Zustimmung Stimmbevölkerung (%)", farbe=None)

## Vergleich aller Akteure

Kongruenz aller Institutionen und Parteien nebeneinander.


Alle Akteure vergleichen und Plot für den Blog speichern.


In [ ]:
akteur_map = {
    'zustimmung_br-pos':   'Bundesrat',
    'zustimmung_bv-pos':   'Bundes-\nversammlung',
    'zustimmung_p-sps':    'SP',
    'zustimmung_p-gps':    'Grüne',
    'zustimmung_p-mitte':  'Mitte',
    'zustimmung_p-fdp':    'FDP',
    'zustimmung_p-svp':    'SVP',
}

akteur_cols = list(akteur_map.keys())

df_long = df[akteur_cols].melt(var_name="akteur", value_name="zustimmung")
df_long['akteur'] = df_long['akteur'].map(akteur_map)

# Optional: feste Reihenfolge für den Plot
df_long['akteur'] = pd.Categorical(df_long['akteur'], categories=akteur_map.values(), ordered=True)

fig = boxplot(df_long, x="akteur", y="zustimmung",
        xlabel="Akteur", ylabel="Kongruenzwert",
        palette=PALETTE_KATEGORIAL_VIELE_WERTE[:8],
        figsize=(8, 4.5),
        width=0.75)

plt.savefig("../Blog/blog_plots/d2_kongruenz_akteur.png",
            dpi=150, bbox_inches='tight', transparent=True)
plt.show()


Mittlere Kongruenz pro Akteur als Tabelle.


In [ ]:
# Mittlerer Kongruenzwert pro Akteur
df[akteur_cols].mean().rename(index=akteur_map).round(3).to_frame('mean_kongruenz')


## Verbände (deaktiviert)

Experiment mit Verbänden – nicht Teil der aktuellen Blog-Auswertung.


Verbände-Code – aktuell auskommentiert (Triple-Quotes).


In [ ]:
"""gruppen = {
    'Bund': ['zustimmung_br-pos', 'zustimmung_bv-pos'],
    'Parteien': ["zustimmung_p-sps", "zustimmung_p-gps", "zustimmung_p-glp", "zustimmung_p-mitte", "zustimmung_p-fdp", "zustimmung_p-svp"]
}

frames = []
for gruppe, cols in gruppen.items():
    temp = df[cols].melt(var_name='organisation', value_name='zustimmung')
    temp['organisation'] = temp['organisation'].str.replace('zustimmung_p-', '')
    temp['gruppe'] = gruppe
    frames.append(temp)

df_verb = pd.concat(frames, ignore_index=True)"""


Prototyp für Verbände – nur wenn `df_verb` oben erzeugt wurde.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(data=df_verb, x='organisation', y='zustimmung',
            hue='gruppe', dodge=False,
            palette={'Bundesrat': '#888888', "Wirtschaft": "#4477AA", "Gewerkschaften": "#DD8899",
                     "Verkehr": "#AAEEDD", "Andere": "#BBAA55"},
            width=0.8, gap=0, ax=ax)

ax.set_title('Zustimmung nach Verbandsgruppen', fontsize=14)
ax.set_xlabel('')
ax.set_ylabel('Zustimmung')
ax.legend(title='Gruppe', loc='upper right')
plt.tight_layout()
plt.show()
